# Prompt chaining and gate checks

Companion notebook for **lesson 4.2**, in the lesson's order: a sequential chain, a hard-coded workflow with an LLM classifier, gate checks (format, content, logic), and a production-style code-writing chain with a retry loop.

Cells marked with a model call need Colab; everything else runs anywhere and shows real output.

**Run it in Colab, setup cell first.** It uses Colab's built-in models (no API key); uncomment a `MODEL = ...` line there to choose one.

In [ ]:
# Setup: Colab's built-in models, no API key needed. Run this cell first.
from google.colab import ai
print("Available models:", ", ".join(ai.list_models()))

MODEL = None                              # Colab's default; uncomment a line below to choose
# MODEL = "google/gemini-2.5-flash"
# MODEL = "google/gemini-2.5-pro"
# MODEL = "google/gemini-3.8-flash"

def get_completion(messages):
    """Send chat messages to the model. Colab's generate_text takes one prompt, so roles are inlined."""
    prompt = "\n\n".join(f"{m['role'].upper()}:\n{m['content']}" for m in messages) + "\n\nASSISTANT:\n"
    return ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)

## 1 · A sequential chain

Each step is one focused prompt; each output becomes part of the next prompt. The API remembers nothing, so the code carries the state.

In [1]:
def ask(prompt: str) -> str:
    return get_completion([{"role": "user", "content": prompt}])

def linkedin_post(topic: str) -> str:
    concepts = ask(f"List the key concepts of {topic}, one line each.")
    summary = ask(f"Summarize these concepts in 5 bullet points:\n\n{concepts}")
    return ask(f"Draft a LinkedIn post under 150 words based on this summary:\n\n{summary}")

In [ ]:
print(linkedin_post("AI agents"))

## 2 · A hard-coded workflow: your code decides

The first LLM call is a yes/no classifier; plain code branches on its answer and calls the tool. `get_calendar` stands in for a real calendar API.

In [2]:
def get_calendar(day: str) -> list[str]:
    """Stand-in for a calendar API."""
    return ["09:00 Dentist (Dr. Lee)", "14:00 Team sync"] if day == "tomorrow" else []

def needs_calendar(question: str) -> bool:
    reply = ask(f"Does answering this question require the user's calendar? Answer only YES or NO.\n\n{question}")
    return reply.strip().upper().startswith("YES")          # normalize: the code branches on this

def answer(question: str) -> str:
    if needs_calendar(question):
        events = "\n".join(get_calendar("tomorrow")) or "no events"
        return ask(f"Calendar for tomorrow:\n{events}\n\nUsing only the calendar, answer: {question}")
    return ask(question)

In [ ]:
for q in ["What time is my dental appointment tomorrow?", "What is the capital of France?"]:
    print(f"Q: {q}\nA: {answer(q)}\n")

## 3 · Gate checks: format and content

A research step must return JSON findings, and every quote must really appear in the source it cites, which catches invented quotes. The gate returns a list of problems (empty = pass), ready to send back in a retry.

In [3]:
import json
from pydantic import BaseModel, ValidationError

class Finding(BaseModel):
    claim: str
    quote: str          # must be copied word-for-word from a source
    source_id: int

def check_research(raw: str, sources: list[str]) -> list[str]:
    try:
        findings = [Finding(**item) for item in json.loads(raw)]
    except ValidationError as e:
        return [f"Format: {'.'.join(map(str, err['loc']))}: {err['msg']}" for err in e.errors()]
    except (json.JSONDecodeError, TypeError) as e:
        return [f"Format: not a JSON list of findings ({e})"]
    return [f"Quote not found in source {f.source_id}: {f.quote!r}"
            for f in findings
            if not (0 <= f.source_id < len(sources)) or f.quote not in sources[f.source_id]]

sources = ["Agents combine an LLM with tools and memory.", "Gate checks validate each step's output."]
for raw in ['[{"claim": "Agents use tools", "quote": "an LLM with tools", "source_id": 0}]',
            '[{"claim": "Gates catch errors", "quote": "gates catch every error", "source_id": 1}]',
            '[{"claim": "Missing source"}]',
            'Here are my findings: ...']:
    print(check_research(raw, sources) or "✅ pass")

✅ pass
["Quote not found in source 1: 'gates catch every error'"]
['Format: quote: Field required', 'Format: source_id: Field required']
['Format: not a JSON list of findings (Expecting value: line 1 column 1 (char 0))']


## 4 · A production-style code chain

Outline → gate 1 (format) → code → gate 2 (syntax + allowed imports, without running the code) → refine with the errors, capped. First the gate on sample code, then the full chain.

In [4]:
import ast, re

TASK = ("a Python script that reads a CSV, calculates the average of a column, "
        "and writes the result to a new CSV")
ALLOWED_IMPORTS = {"csv", "statistics", "sys"}

def strip_fences(text: str) -> str:
    match = re.search(r"`{3}[ \t]*(?:python|py)?[ \t]*\n(.*?)`{3}", text, re.S)
    return (match.group(1) if match else text).strip()

def check_code(code: str) -> list[str]:
    """Gate 2: valid Python, and only allowed imports. Never runs the code."""
    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        return [f"SyntaxError line {e.lineno}: {e.msg}"]
    imported = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            imported |= {alias.name.split(".")[0] for alias in node.names}
        elif isinstance(node, ast.ImportFrom) and node.module:
            imported.add(node.module.split(".")[0])
    return [f"Restricted import: {name}" for name in sorted(imported - ALLOWED_IMPORTS)]

for sample in ["import csv, statistics\nprint('ok')",
               "import pandas as pd\nimport os.path",
               "def broken(:\n    pass"]:
    print(f"{sample.splitlines()[0]:28} → {check_code(sample) or '✅ pass'}")

import csv, statistics       → ✅ pass
import pandas as pd          → ['Restricted import: os', 'Restricted import: pandas']
def broken(:                 → ['SyntaxError line 1: invalid syntax']


In [5]:
def write_script(max_refines: int = 3) -> str:
    outline = ask(f"You are a helpful programming assistant. I need {TASK}. "
                  "Give a short numbered outline of the steps.")
    if not re.search(r"^\s*1[.)]", outline, re.M) or "read" not in outline.lower():   # gate 1: format
        raise RuntimeError(f"Gate 1 failed: not a numbered outline that mentions reading:\n{outline}")

    code = strip_fences(ask(f"Write the Python script for this outline, using only {sorted(ALLOWED_IMPORTS)}:\n\n{outline}"))
    for attempt in range(max_refines + 1):
        errors = check_code(code)                                                       # gate 2: logic
        print(f"attempt {attempt}: {errors or 'passed gate 2'}")
        if not errors:
            return code
        if attempt == max_refines:
            raise RuntimeError(f"Still failing after {max_refines} refinements: {errors}")
        code = strip_fences(ask(f"This code has problems:\n{code}\n\nProblems:\n{errors}\n\nReturn the corrected script."))

In [ ]:
print(write_script())